# Etterforskning

Vi har en stor ferdiglaget graf, med titusenvis av noder.  Vi skal bruke Neo4J til å lete etter kriminalitet.


In [ ]:
%%bash
PWD=$(pwd)
mkdir -p neo4j
mkdir -p neo4j/data
mkdir -p neo4j/logspo
mkdir -p neo4j/plugins
mkdir -p neo4j/import

# Dette laster ned to plugins fra Neo4j (som må importeres for hånd om vi skal kjøre på VDI)

podman run \
    -p 7474:7474 -p 7687:7687 \
    --userns=keep-id \
    -e NEO4J_PLUGINS='["apoc", "graph-data-science"]' \
    -e NEO4J_dbms_security_procedures_unrestricted='gds.*,apoc.*' \
    -e NEO4J_dbms_security_procedures_allowlist='gds.*,apoc.*' \
    -e NEO4J_apoc_import_file_enabled=true \
    -v $PWD/neo4j/data:/data:Z \
    -v $PWD/neo4j/logs:/logs:Z \
    -v $PWD/neo4j/import:/import:Z \
    -v $PWD/neo4j/plugins:/plugins:Z \
    -e NEO4J_AUTH=neo4j/password \
    -d docker.io/library/neo4j:latest

In [15]:
# Få Kontakt
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Kontakt!")

Kontakt!


In [ ]:
# Sjekke APOC
records, summary, keys = driver.execute_query(
    """RETURN apoc.version() AS version;""")

print(f"Server Address: {summary.server.address}")
print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

Server Address: 127.0.0.1:7687
Keys:
	version: <Record version='2025.11.2'>
Ressursbruk
	Kjøringen: 43ms
	Å konsumere: 0ms


In [4]:
# Sjekke GDS
records, summary, keys = driver.execute_query(
    """
    CALL gds.version();
    """)

print("Keys:")
for k in range(len(keys)):
    print(f"\t{keys[k]}: {records[k]}")
#

Keys:
	gdsVersion: <Record gdsVersion='2.24.0'>


Vi flytter grafen vi skal arbeide med

In [7]:
%%bash
cp grafer/Komplett.graphml neo4j/import

In [16]:
# Ikke starte med gamle data noe sted
records, summary, keys = driver.execute_query(
    """CALL gds.graph.drop('cliqueFinderGraph', false) YIELD graphName"""
)
records, summary, keys = driver.execute_query(
    """match (n) detach delete n""")
print("OK")

OK


In [17]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """CALL apoc.import.graphml("Komplett.graphml", {storeNodeIds: true, readLabels: true})"""
    )
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Grafen")
print(f"\tNye noder: {summary.counters.nodes_created}")
print(f"\tNye kanter: {summary.counters.relationships_created}")
print("Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder")
      
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	file: Komplett.graphml
	source: file
	format: graphml
	nodes: 56803
	relationships: 103420
	properties: 1023
	time: 1105
	rows: 0
	batchSize: -1
	batches: 0
	done: True
	data: None
Grafen
	Nye noder: 0
	Nye kanter: 0
Begge er 0 fordi dette bare gir mening når Cypher koden Per Se genererer noder
Ressursbruk
	Kjøringen: 5ms
	Å konsumere: 1108ms


## Lete etter et esel
Vi ser etter en klikk hvor alle kjenner hverandre.  Vi finner kanskje mange.  Det er interessant om (nesten) alle (eller i det minste flertallet) har uttak av kontanter.
Dette er typisk GDS-mat (algoritmer som kjører på hele grafen, ikke bare på enkeltnoder).

Grafen er stor, og vi ønsker å gjøre forberedelser.  Det første er å merke alle noder som har relasjoner av en gitt type (for oss: *Kjenner*).  Det vil si at der en node har en kjenner-relasjon til noen, da merker vi noden med en "har venner"-egenskap.  Dermed behøver vi ikke se på kantene når vi søker, og alt går raskere.

Om en slik metode har verdi avhenger av bruken.  Antallet her er ikke så stort, for det er bare "de kriminelle" som har denne egenskapen.

In [ ]:
# Finn alle personer som har venner, og sett på en egenskap
records, summary, keys = driver.execute_query(
    """
    MATCH (n)
    WHERE (n)-[:Kjenner]-()
    SET n.harVenner = 1     // (altså true)
    RETURN COUNT(n) AS ANTALL
    """
)
print(f"Antall satt på: {summary.counters.properties_set}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

Antall satt på: 78
Ressursbruk
	Kjøringen: 56ms
	Å konsumere: 28ms


In [20]:
# Da henter vi inn en sub-graf, 
records, summary, keys = driver.execute_query(
    """CALL 
    gds.graph.project(
      'cliqueFinderGraph',      // Navnet på sub-grafen i minnet
      {
        Person: {             // Plukke ut nodene
          label: 'Person',
          properties: 'harVenner'
        }
      },
      {
        Kjenner: {            // Plukke ut kantene
          type: 'Kjenner',
          orientation: 'UNDIRECTED'  // Dersom det skulle være retning ser vi bort fra det
      }
    }
  )
  """
)
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

ClientError: {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.UnsupportedOperationException: Loading of values of type Boolean is currently not supported} {gql_status: 52N37} {gql_status_description: error: procedure exception - procedure execution error. Execution of the procedure gds.graph.project() failed.}

In [15]:
# Lese inn grafen vi har bygget tidligere
records, summary, keys = driver.execute_query(
    """CALL 
    gds.kcore.stream('cliqueFinderGraph')
    YIELD nodeId, coreValue
    RETURN gds.util.asNode(nodeId).id AS id, coreValue
    ORDER BY coreValue DESC
    LIMIT 20
    """
)
# for enkelt å pakke opp svaret
for record in records:
    record_dict = record.data()
#
for k in record_dict:
    print(f"\t{k}: {record_dict[k]}")
#
print("Ressursbruk")
print(f"\tKjøringen: {summary.result_available_after}ms")
print(f"\tÅ konsumere: {summary.result_consumed_after}ms")

	id: eaae69d0-473f-4089-ac74-83705d738881
	coreValue: 9
Ressursbruk
	Kjøringen: 33ms
	Å konsumere: 24ms
